In [1]:
import os, csv, pandas as pd

In [2]:
sumlev_names={'010':'US','020':'Region','030':'Division','040':'State','050':'County',
             '060':'County Subdiv','140':'Tract','150':'Blk Group','160':'Place','170':'Cons City',
              '250':'Tribal Area','251':'Tribal Subdiv','256':'Tribal Tract','258':'Tribal Blk Group', 
              '310':'CBSA','400':'Urban Area','500':'Cong Dist','795':'PUMA','860':'ZCTA'}

In [3]:
geo_names={}
name_file=os.path.join('metatables','acs_geos.csv')
with open(name_file, 'r', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        geo_names[row['geoid']]=row['geoname']
len(geo_names)

405898

# Significant Difference

In [4]:
geo_file=os.path.join('geosd_count.json')
geo_sd=pd.read_json(geo_file,orient='index')
geo_sd['geoname']=geo_sd.index.map(geo_names)
geo_sd['sumlevel']=geo_sd.index.str.slice(0, 3)
geo_sd['sumlevel']=geo_sd['sumlevel'].map(sumlev_names)
geo_sd.sort_values(by='true_pct',ascending=False)

,true,false,null,true_pct,false_pct,null_pct,geoname,sumlevel
02000US4,303,15,0,95.3,4.7,0.0,West Region,Region
03000US5,302,16,0,95.0,5.0,0.0,South Atlantic Division,Division
01000US,299,19,0,94.0,6.0,0.0,United States,US
02000US3,296,22,0,93.1,6.9,0.0,South Region,Region
06000US4715790158,294,24,0,92.5,7.5,0.0,"District 1, Shelby County, Tennessee",County Subdiv
...,...,...,...,...,...,...,...,...
14000US17031980100,0,308,10,0.0,96.9,3.1,"Census Tract 9801, Cook County, Illinois",Tract
14000US17031980000,0,308,10,0.0,96.9,3.1,"Census Tract 9800, Cook County, Illinois",Tract
15000US360610084001,0,227,91,0.0,71.4,28.6,"Block Group 1, Census Tract 84, New York Count...",Blk Group
15000US360610086010,0,227,91,0.0,71.4,28.6,"Block Group 0, Census Tract 86.01, New York Co...",Blk Group


In [5]:
geocount=len(geo_sd)
geocount

405787

In [6]:
sum_geotrue=geo_sd['true'].sum()
sum_geotrue

11648350

In [7]:
min_true = geo_sd['true'].min()
max_true = geo_sd['true'].max()
mean_true = round(geo_sd['true'].mean(),0)
std_true=round(geo_sd['true'].std(),0)
med_true = geo_sd['true'].median()
mode_true = geo_sd['true'].mode().iloc[0] 
print(min_true,max_true,mean_true,std_true,med_true,mode_true)

0 303 29.0 23.0 23.0 15


In [8]:
bins = [-1, 0, 24.9, 49.9, 74.9, 100]
labels=['0','0.1 to 24.9', '25 to 49.9', '50 to 74.9', '75 to 100']
geo_sd['true_bin']=pd.cut(geo_sd['true_pct'],bins=bins,labels=labels,include_lowest=True)
geo_sd.sort_values(by='true_pct',ascending=False)

,true,false,null,true_pct,false_pct,null_pct,geoname,sumlevel,true_bin
02000US4,303,15,0,95.3,4.7,0.0,West Region,Region,75 to 100
03000US5,302,16,0,95.0,5.0,0.0,South Atlantic Division,Division,75 to 100
01000US,299,19,0,94.0,6.0,0.0,United States,US,75 to 100
02000US3,296,22,0,93.1,6.9,0.0,South Region,Region,75 to 100
06000US4715790158,294,24,0,92.5,7.5,0.0,"District 1, Shelby County, Tennessee",County Subdiv,75 to 100
...,...,...,...,...,...,...,...,...,...
14000US17031980100,0,308,10,0.0,96.9,3.1,"Census Tract 9801, Cook County, Illinois",Tract,0
14000US17031980000,0,308,10,0.0,96.9,3.1,"Census Tract 9800, Cook County, Illinois",Tract,0
15000US360610084001,0,227,91,0.0,71.4,28.6,"Block Group 1, Census Tract 84, New York Count...",Blk Group,0
15000US360610086010,0,227,91,0.0,71.4,28.6,"Block Group 0, Census Tract 86.01, New York Co...",Blk Group,0


In [40]:
#counties_sd=geo_sd.loc[geo_sd['sumlevel']=='County']
#counties_sd.to_csv('county_counts_sd.csv')
#tracts_sd=geo_sd.loc[geo_sd['sumlevel']=='Tract']
#tracts_sd.to_csv('tracts_counts_sd.csv')

In [9]:
cnt_geotrue_bins=geo_sd.groupby('true_bin',observed=True)['true'].count().reset_index()
cnt_geotrue_bins['pct']=round((cnt_geotrue_bins['true']/geocount)*100,1)
cnt_geotrue_bins

,true_bin,true,pct
0,0,4550,1.1
1,0.1 to 24.9,386962,95.4
2,25 to 49.9,13129,3.2
3,50 to 74.9,1060,0.3
4,75 to 100,86,0.0


In [10]:
geo_sd.loc[geo_sd['true_bin']=='75 to 100',['true','true_pct','geoname']].sort_values(by='true_pct', ascending=False).head(n=47)

,true,true_pct,geoname
02000US4,303,95.3,West Region
03000US5,302,95.0,South Atlantic Division
01000US,299,94.0,United States
02000US3,296,93.1,South Region
03000US8,294,92.5,Mountain Division
06000US4715790158,294,92.5,"District 1, Shelby County, Tennessee"
04000US06,292,91.8,California
02000US2,291,91.5,Midwest Region
03000US7,289,90.9,West South Central Division
03000US9,289,90.9,Pacific Division


In [11]:
cnt_sumlevs=geo_sd.groupby(['true_bin','sumlevel'],observed=True)['true'].count().reset_index()
cnt_sumlevs.sort_values(by=['true_bin','true'],ascending=False)

,true_bin,sumlevel,true
56,75 to 100,State,26
50,75 to 100,CBSA,18
58,75 to 100,Urban Area,13
53,75 to 100,Division,9
51,75 to 100,County,8
52,75 to 100,County Subdiv,5
55,75 to 100,Region,4
54,75 to 100,Place,2
57,75 to 100,US,1
39,50 to 74.9,Cong Dist,303


In [12]:
cnt_sumlevs['pct_group']=round((cnt_sumlevs['true']/cnt_sumlevs.groupby(
    ['true_bin'],observed=True)['true'].transform('sum'))*100,1)
cnt_sumlevs.sort_values(by=['true_bin','pct_group','true'],ascending=False)

,true_bin,sumlevel,true,pct_group
56,75 to 100,State,26,30.2
50,75 to 100,CBSA,18,20.9
58,75 to 100,Urban Area,13,15.1
53,75 to 100,Division,9,10.5
51,75 to 100,County,8,9.3
52,75 to 100,County Subdiv,5,5.8
55,75 to 100,Region,4,4.7
54,75 to 100,Place,2,2.3
57,75 to 100,US,1,1.2
39,50 to 74.9,Cong Dist,303,28.6


# CV

In [13]:
geo_file_cv=os.path.join('geocv_count.json')
geo_cv=pd.read_json(geo_file_cv,orient='index')
geo_cv['medhigh']=geo_cv['medium']+geo_cv['high']
geo_cv['medhigh_pct']=geo_cv['medium_pct']+geo_cv['high_pct']
geo_cv['geoname']=geo_cv.index.map(geo_names)
geo_cv['sumlevel']=geo_cv.index.str.slice(0, 3)
geo_cv['sumlevel']=geo_cv['sumlevel'].map(sumlev_names)
geo_cv.sort_values(by='medhigh',ascending=False)

,low,medium,high,null,low_pct,medium_pct,high_pct,null_pct,medhigh,medhigh_pct,geoname,sumlevel
01000US,7,29,263,0,2.3,9.7,88.0,0.0,292,97.7,United States,US
02000US3,13,44,239,0,4.4,14.9,80.7,0.0,283,95.6,South Region,Region
02000US4,27,38,238,0,8.9,12.5,78.5,0.0,276,91.0,West Region,Region
03000US5,27,50,225,0,8.9,16.6,74.5,0.0,275,91.1,South Atlantic Division,Division
06000US4715790158,28,72,194,0,9.5,24.5,66.0,0.0,266,90.5,"District 1, Shelby County, Tennessee",County Subdiv
...,...,...,...,...,...,...,...,...,...,...,...,...
14000US39109365301,35,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Census Tract 3653.01, Miami County, Ohio",Tract
15000US290693601001,13,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Block Group 1, Census Tract 3601, Dunklin Coun...",Blk Group
15000US290693601002,14,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Block Group 2, Census Tract 3601, Dunklin Coun...",Blk Group
15000US290693601003,10,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Block Group 3, Census Tract 3601, Dunklin Coun...",Blk Group


In [14]:
sum_medhigh=geo_cv['medhigh'].sum()
sum_medhigh

1030443

In [15]:
min_mh = geo_cv['medhigh'].min()
max_mh = geo_cv['medhigh'].max()
mean_mh = round(geo_cv['medhigh'].mean(),0)
std_mh=round(geo_cv['medhigh'].std(),0)
med_mh = geo_cv['medhigh'].median()
mode_mh = geo_cv['medhigh'].mode().iloc[0] 
print(min_mh, max_mh, mean_mh,std_mh,med_mh,mode_mh)

0 292 3.0 10.0 0.0 0


In [16]:
bins = [-1, 0, 74, 149, 224, 300]
labels=['0','1 to 74', '75 to 149', '150 to 224', '225 to 300']
geo_cv['medhigh_bins']=pd.cut(geo_cv['medhigh'],bins=bins,labels=labels,include_lowest=True)
geo_cv.sort_values(by='medhigh',ascending=False)

,low,medium,high,null,low_pct,medium_pct,high_pct,null_pct,medhigh,medhigh_pct,geoname,sumlevel,medhigh_bins
01000US,7,29,263,0,2.3,9.7,88.0,0.0,292,97.7,United States,US,225 to 300
02000US3,13,44,239,0,4.4,14.9,80.7,0.0,283,95.6,South Region,Region,225 to 300
02000US4,27,38,238,0,8.9,12.5,78.5,0.0,276,91.0,West Region,Region,225 to 300
03000US5,27,50,225,0,8.9,16.6,74.5,0.0,275,91.1,South Atlantic Division,Division,225 to 300
06000US4715790158,28,72,194,0,9.5,24.5,66.0,0.0,266,90.5,"District 1, Shelby County, Tennessee",County Subdiv,225 to 300
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14000US39109365301,35,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Census Tract 3653.01, Miami County, Ohio",Tract,0
15000US290693601001,13,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Block Group 1, Census Tract 3601, Dunklin Coun...",Blk Group,0
15000US290693601002,14,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Block Group 2, Census Tract 3601, Dunklin Coun...",Blk Group,0
15000US290693601003,10,0,0,0,100.0,0.0,0.0,0.0,0,0.0,"Block Group 3, Census Tract 3601, Dunklin Coun...",Blk Group,0


In [41]:
#counties_cv=geo_cv.loc[geo_cv['sumlevel']=='County']
#counties_cv.to_csv('county_counts_cv.csv')
#tracts_cv=geo_cv.loc[geo_cv['sumlevel']=='Tract']
#tracts_cv.to_csv('tracts_counts_cv.csv')

In [17]:
cnt_medhigh_bins=geo_cv.groupby('medhigh_bins',observed=True)['medhigh'].count().reset_index()
cnt_medhigh_bins['pct']=round((cnt_medhigh_bins['medhigh']/geocount)*100,1)
cnt_medhigh_bins

,medhigh_bins,medhigh,pct
0,0,252157,62.1
1,1 to 74,152281,37.5
2,75 to 149,1176,0.3
3,150 to 224,150,0.0
4,225 to 300,23,0.0


In [18]:
geo_cv.loc[geo_cv['medhigh_bins']=='225 to 300',['medhigh','medhigh_pct','geoname']].sort_values(by=['medhigh_pct','medhigh'], ascending=False).head(n=20)

,medhigh,medhigh_pct,geoname
01000US,292,97.7,United States
02000US3,283,95.6,South Region
03000US9,266,92.1,Pacific Division
03000US5,275,91.1,South Atlantic Division
02000US4,276,91.0,West Region
02000US2,264,90.8,Midwest Region
06000US4715790158,266,90.5,"District 1, Shelby County, Tennessee"
03000US7,260,89.9,West South Central Division
04000US06,262,89.7,California
03000US8,262,89.1,Mountain Division


In [19]:
geo_cv.loc[(geo_cv['medhigh']<5)&(geo_cv['medhigh']!=0)]['medhigh'].count()

112843

In [20]:
geo_cv.loc[(geo_cv['medhigh']>=5)&(geo_cv['medhigh']!=0)]['medhigh'].count()

40787

In [21]:
cnt_medhigh_bins_subset=geo_cv.loc[geo_cv['medhigh']>=5].groupby('medhigh_bins',observed=True)['medhigh'].count().reset_index()
cnt_medhigh_bins_subset['pct']=round((cnt_medhigh_bins['medhigh']/geocount)*100,1)
cnt_medhigh_bins_subset

,medhigh_bins,medhigh,pct
0,1 to 74,39438,62.1
1,75 to 149,1176,37.5
2,150 to 224,150,0.3
3,225 to 300,23,0.0


In [22]:
cnt_cv_sumlevs=geo_cv.groupby(['medhigh_bins','sumlevel'],observed=True)['medhigh'].count().reset_index()
cnt_cv_sumlevs.sort_values(by=['medhigh_bins','medhigh'],ascending=False).head(n=50)

,medhigh_bins,sumlevel,medhigh
50,225 to 300,Division,7
52,225 to 300,State,5
51,225 to 300,Region,4
49,225 to 300,County Subdiv,2
54,225 to 300,Urban Area,2
47,225 to 300,CBSA,1
48,225 to 300,County,1
53,225 to 300,US,1
41,150 to 224,County,32
39,150 to 224,CBSA,31


In [23]:
cnt_cv_sumlevs['pct_group']=round((cnt_cv_sumlevs['medhigh']/cnt_cv_sumlevs.groupby(
    ['medhigh_bins'],observed=True)['medhigh'].transform('sum'))*100,1)
cnt_cv_sumlevs.sort_values(by=['medhigh_bins','pct_group','medhigh'],ascending=False)

,medhigh_bins,sumlevel,medhigh,pct_group
50,225 to 300,Division,7,30.4
52,225 to 300,State,5,21.7
51,225 to 300,Region,4,17.4
49,225 to 300,County Subdiv,2,8.7
54,225 to 300,Urban Area,2,8.7
47,225 to 300,CBSA,1,4.3
48,225 to 300,County,1,4.3
53,225 to 300,US,1,4.3
41,150 to 224,County,32,21.3
39,150 to 224,CBSA,31,20.7
